# LLM-Augmented Hybrid — XGBoost Methods Across Three Seeds

Clean public release for the MAPR 2026 paper **LLM-Augmented Hybrid Representations for Disease Category Classification from Clinical Notes**.

- Fixed split seed: `42`
- Experiment seeds: `42`, `123`, and `456`
- Dataset: credentialed MIMIC-IV-Ext-DiReCT v1.0.0
- Saved outputs are intentionally cleared from this release.


## Setup

This cell mounts Google Drive when running in Colab and installs only `rarfile`, matching the old notebook setup style.

Colab's preinstalled ML/data package versions are intentionally preserved to reduce result drift. Do not upgrade `xgboost`, `scikit-learn`, `transformers`, `sentence-transformers`, `pandas`, `numpy`, `torch`, `openai`, or related experiment packages unless the team records and approves the exact version change.

No system package installation is performed. A later audit cell prints package versions after imports so the run environment is recorded.


In [ ]:
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Not in Colab; using local paths.')

if IN_COLAB:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', *['rarfile==4.2',
 'openai==2.32.0',
 'scikit-learn==1.6.1',
 'xgboost==3.2.0',
 'transformers==5.0.0',
 'sentence-transformers==5.4.1']],
        check=True,
    )


## Configuration And Paths

This cell defines the fixed split seed, rerun seeds, prompt name, model names, and project paths.

Default Colab paths:

- Project root: `/content/drive/MyDrive/NCKH`
- Data root: `/content/drive/MyDrive/NCKH/data/`
- Cache root: `/content/drive/MyDrive/NCKH/cache/rerun_3seeds/`
- Results root: `/content/drive/MyDrive/NCKH/results/`

When running locally, the notebook falls back to the current folder for project data and results. The same code should therefore work both in Colab and in this downloaded local folder.


In [ ]:
import os, json, time, random, hashlib, concurrent.futures
from pathlib import Path
import rarfile
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report
import xgboost as xgb
import openai

SPLIT_SEED = 42
RUN_SEEDS = [42, 123, 456]
PROMPT_NAME = 'Prompt_V3'
OPENAI_MODEL = 'gpt-4o-mini'
OPENAI_TEMPERATURE = 0.0
EXTERNAL_LLM_DATA_COMPLIANCE_ACKNOWLEDGED = False
OPENAI_MAX_TOKENS = 500

if IN_COLAB:
    PROJECT_ROOT = Path(os.environ.get('LLM_FEATURES_PROJECT_ROOT', '/content/drive/MyDrive/LLM-features-clinical-notes'))
    EXTRACTED_ROOT = Path('/content/extracted_data')
else:
    PROJECT_ROOT = Path.cwd()
    EXTRACTED_ROOT = Path('/tmp/extracted_data')

DATA_ROOT = PROJECT_ROOT / 'data'
SPLIT_DIR = DATA_ROOT / 'splits'
CACHE_ROOT = PROJECT_ROOT / 'cache' / 'rerun_3seeds'
RESULTS_ROOT = PROJECT_ROOT / 'results'
for p in [SPLIT_DIR, CACHE_ROOT, RESULTS_ROOT, EXTRACTED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('IN_COLAB:', IN_COLAB)
print('DATA_ROOT:', DATA_ROOT)
print('CACHE_ROOT:', CACHE_ROOT)
print('RESULTS_ROOT:', RESULTS_ROOT)

In [ ]:
def print_package_versions_for_audit():
    import importlib
    import platform

    package_modules = {
        'numpy': 'numpy',
        'pandas': 'pandas',
        'scikit-learn': 'sklearn',
        'xgboost': 'xgboost',
        'torch': 'torch',
        'transformers': 'transformers',
        'sentence-transformers': 'sentence_transformers',
        'openai': 'openai',
        'tqdm': 'tqdm',
        'rarfile': 'rarfile',
    }
    print('Package versions for audit:')
    print(f'  python: {platform.python_version()}')
    for package_name, module_name in package_modules.items():
        module = importlib.import_module(module_name)
        print(f"  {package_name}: {getattr(module, '__version__', 'unknown')}")


print_package_versions_for_audit()


## External LLM Access and Data Compliance

This notebook can send clinical-note text to an external LLM service. Before running the next cell, verify that your use complies with the current PhysioNet agreement and institutional requirements, including applicable zero-data-retention, no-training, and no-human-review protections.

After completing that review, set `EXTERNAL_LLM_DATA_COMPLIANCE_ACKNOWLEDGED = True` in the configuration cell. Store `OPENAI_API_KEY` in an environment variable or Colab Secrets; never paste it into the notebook.


In [ ]:
def require_external_llm_data_compliance():
    if not EXTERNAL_LLM_DATA_COMPLIANCE_ACKNOWLEDGED:
        raise RuntimeError(
            'Data-compliance acknowledgement is required before external LLM calls. '
            'Review the README and current PhysioNet guidance, then set '
            'EXTERNAL_LLM_DATA_COMPLIANCE_ACKNOWLEDGED = True in a separate cell.'
        )


require_external_llm_data_compliance()


def load_openai_api_key():
    key = os.environ.get('OPENAI_API_KEY')
    if key:
        return key
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get('OPENAI_API_KEY')
        except Exception:
            return None
    return None

OPENAI_API_KEY = load_openai_api_key()
if not OPENAI_API_KEY:
    raise RuntimeError('OPENAI_API_KEY is not set. In Colab, add it to Secrets as OPENAI_API_KEY.')
openai.api_key = OPENAI_API_KEY
openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)
print('OpenAI API key loaded.')


## Data Extraction, Fixed Split, And Text Loading

## Data Extraction and Exact Split Reconstruction

The clinical data are not distributed with this repository. Acquire MIMIC-IV-Ext-DiReCT v1.0.0 through PhysioNet and place `samples.rar` under `data/`.

The original experiment fixed `random_state=42`. Its input rows came from an unsorted Google Colab `os.walk`, whose traversal order is filesystem-dependent. The code below reconstructs that exact traversal using a 511-index permutation over a canonical lexicographic scan. This corrects a reproducibility issue; it does **not** select a favorable seed or alter the reported split.

The notebook validates the acquired file manifest, reconstructs the same two-stage stratified split, and checks the reported 308/101/102 sample counts and 25/23/23 category counts. Generated split CSVs remain local and are ignored by Git.


In [ ]:
def extract_samples_if_needed():
    rar_path = DATA_ROOT / 'samples.rar'
    if not rar_path.exists():
        raise FileNotFoundError(
            f'Missing {rar_path}. Acquire MIMIC-IV-Ext-DiReCT v1.0.0 from PhysioNet first.'
        )
    if (EXTRACTED_ROOT / 'Finished').exists() or (EXTRACTED_ROOT / 'samples').exists():
        print('Using existing extracted samples:', EXTRACTED_ROOT)
        return
    print('Extracting samples:', rar_path)
    with rarfile.RarFile(str(rar_path), 'r') as rf:
        rf.extractall(str(EXTRACTED_ROOT))


def find_finished_dir():
    for candidate in [EXTRACTED_ROOT / 'Finished', EXTRACTED_ROOT / 'samples']:
        if candidate.exists():
            return candidate
    for root, dirs, files in os.walk(EXTRACTED_ROOT):
        root_path = Path(root)
        if root_path.name in {'Finished', 'samples'}:
            return root_path
    raise FileNotFoundError(f'Cannot find Finished/ or samples/ under {EXTRACTED_ROOT}')


extract_samples_if_needed()
FINISHED_DIR = find_finished_dir()
print('FINISHED_DIR:', FINISHED_DIR)


# The original experiment fixed random_state=42 before evaluation. However,
# os.walk does not guarantee directory-entry order, so the same seed can yield
# different members when the input row order changes across filesystems.
#
# This 511-index permutation records the original Google Colab traversal order
# relative to a canonical lexicographic scan. It contains no clinical text,
# filenames, subject identifiers, labels, or precomputed split membership.
EXPECTED_SAMPLE_COUNT = 511
EXPECTED_CANONICAL_MANIFEST_SHA256 = 'd5173feb1a6ed1823c0723192af08862135b36349a261bc88747178575e7aa58'
COLAB_OS_WALK_ORDER_INDICES = [302, 301, 94, 91, 92, 87, 88, 85, 90, 86, 89, 93, 234, 239, 237, 216, 248, 236, 244, 242, 222,
 235, 224, 232, 231, 220, 221, 227, 246, 218, 229, 247, 211, 226, 208, 238, 213, 219, 217, 245,
 223, 243, 214, 212, 228, 240, 215, 230, 241, 209, 233, 210, 225, 65, 66, 67, 80, 77, 79, 76,
 75, 83, 84, 82, 78, 81, 71, 74, 72, 69, 68, 70, 73, 327, 318, 315, 319, 320, 317, 305, 312,
 304, 332, 330, 306, 309, 334, 307, 323, 322, 325, 308, 321, 314, 326, 331, 333, 311, 316, 324,
 310, 313, 329, 328, 303, 414, 408, 421, 409, 406, 407, 412, 410, 420, 413, 417, 411, 416, 415,
 418, 419, 425, 424, 422, 423, 480, 476, 487, 482, 483, 484, 481, 478, 477, 486, 479, 485, 488,
 462, 468, 463, 461, 467, 465, 464, 470, 474, 472, 471, 473, 469, 475, 466, 338, 337, 335, 336,
 265, 266, 293, 281, 251, 272, 249, 257, 261, 285, 283, 296, 256, 258, 262, 286, 300, 277, 252,
 254, 275, 270, 273, 282, 284, 290, 268, 288, 297, 289, 259, 263, 274, 278, 280, 253, 298, 291,
 271, 299, 255, 260, 264, 276, 294, 250, 295, 279, 292, 287, 267, 269, 109, 110, 111, 114, 120,
 121, 118, 119, 112, 113, 116, 115, 117, 447, 448, 459, 456, 458, 455, 454, 449, 457, 453, 452,
 460, 451, 450, 428, 444, 435, 431, 443, 438, 446, 437, 439, 427, 441, 429, 430, 426, 445, 433,
 436, 440, 442, 434, 432, 499, 500, 501, 503, 502, 339, 340, 341, 354, 357, 342, 353, 362, 359,
 347, 343, 349, 346, 351, 355, 360, 358, 361, 350, 345, 348, 356, 344, 352, 365, 364, 363, 134,
 148, 140, 137, 136, 139, 150, 149, 133, 138, 141, 135, 146, 145, 144, 132, 143, 147, 142, 169,
 167, 172, 171, 168, 170, 162, 164, 165, 160, 161, 163, 166, 510, 508, 509, 507, 506, 505, 504,
 386, 381, 393, 374, 378, 385, 384, 375, 392, 382, 388, 387, 379, 389, 377, 391, 380, 376, 390,
 383, 373, 370, 366, 369, 371, 368, 367, 372, 405, 397, 396, 403, 401, 404, 402, 400, 394, 395,
 399, 398, 496, 497, 498, 489, 490, 491, 492, 493, 495, 494, 130, 131, 127, 128, 124, 125, 129,
 122, 126, 123, 103, 104, 107, 101, 100, 105, 102, 106, 108, 96, 97, 99, 95, 98, 202, 195, 201,
 197, 200, 198, 196, 199, 206, 204, 205, 207, 203, 184, 181, 186, 191, 188, 182, 185, 192, 193,
 183, 190, 187, 194, 189, 176, 180, 177, 179, 178, 174, 173, 175, 59, 63, 55, 47, 44, 49, 51,
 62, 56, 61, 53, 50, 52, 64, 54, 57, 60, 43, 58, 46, 45, 48, 10, 9, 7, 3, 16, 22, 13, 2, 14, 23,
 17, 0, 21, 8, 19, 12, 4, 15, 1, 24, 18, 6, 26, 27, 20, 25, 11, 5, 35, 29, 42, 34, 38, 37, 36,
 41, 32, 33, 30, 39, 31, 40, 28, 156, 157, 151, 154, 153, 155, 152, 158, 159]


def _relative_path_from_file_path(file_path):
    normalized = str(file_path).replace('\\', '/')
    for marker in ['/Finished/', '/samples/']:
        if marker in normalized:
            return normalized.split(marker, 1)[1]
    return normalized


def collect_json_files_in_reported_order(finished_dir):
    canonical_rows = []
    finished_dir = Path(finished_dir)
    for root, dirs, files in os.walk(finished_dir):
        dirs.sort()
        for file_name in sorted(files):
            if not file_name.endswith('.json'):
                continue
            file_path = Path(root) / file_name
            relative_path = file_path.relative_to(finished_dir).as_posix()
            canonical_rows.append({
                'relative_path': relative_path,
                'file_path': str(file_path),
                'section': relative_path.split('/')[0],
            })
    canonical_rows.sort(key=lambda row: row['relative_path'])

    relative_paths = [row['relative_path'] for row in canonical_rows]
    if len(relative_paths) != EXPECTED_SAMPLE_COUNT:
        raise RuntimeError(
            f'Expected {EXPECTED_SAMPLE_COUNT} JSON files, found {len(relative_paths)}. '
            'Check that MIMIC-IV-Ext-DiReCT v1.0.0 was acquired and extracted correctly.'
        )
    if len(set(relative_paths)) != EXPECTED_SAMPLE_COUNT:
        raise RuntimeError('Duplicate relative paths found in the acquired dataset')

    manifest_hash = hashlib.sha256('\n'.join(relative_paths).encode('utf-8')).hexdigest()
    if manifest_hash != EXPECTED_CANONICAL_MANIFEST_SHA256:
        raise RuntimeError(
            'The acquired file manifest does not match the dataset version used in the paper. '
            f'Expected SHA-256 {EXPECTED_CANONICAL_MANIFEST_SHA256}, found {manifest_hash}.'
        )

    if len(COLAB_OS_WALK_ORDER_INDICES) != EXPECTED_SAMPLE_COUNT:
        raise RuntimeError('Recovered Colab order has the wrong length')
    if sorted(COLAB_OS_WALK_ORDER_INDICES) != list(range(EXPECTED_SAMPLE_COUNT)):
        raise RuntimeError('Recovered Colab order must contain each index from 0 to 510 exactly once')

    ordered_rows = [canonical_rows[index] for index in COLAB_OS_WALK_ORDER_INDICES]
    return pd.DataFrame([
        {'file_path': row['file_path'], 'section': row['section']}
        for row in ordered_rows
    ])


def create_fixed_split(seed=SPLIT_SEED):
    df_files = collect_json_files_in_reported_order(FINISHED_DIR)

    section_counts_initial = df_files['section'].value_counts()
    sections_for_train_only_initial = section_counts_initial[
        section_counts_initial < 3
    ].index.tolist()

    df_splittable_60_40 = df_files[
        ~df_files['section'].isin(sections_for_train_only_initial)
    ]
    df_non_stratifiable_initial = df_files[
        df_files['section'].isin(sections_for_train_only_initial)
    ]

    train_60, temp_40 = train_test_split(
        df_splittable_60_40,
        test_size=0.4,
        stratify=df_splittable_60_40['section'],
        random_state=seed,
    )

    section_counts_temp_40 = temp_40['section'].value_counts()
    sections_for_train_only_from_temp = section_counts_temp_40[
        section_counts_temp_40 < 2
    ].index.tolist()

    temp_40_splittable_50_50 = temp_40[
        ~temp_40['section'].isin(sections_for_train_only_from_temp)
    ]
    df_non_stratifiable_from_temp = temp_40[
        temp_40['section'].isin(sections_for_train_only_from_temp)
    ]

    val_20, test_20 = train_test_split(
        temp_40_splittable_50_50,
        test_size=0.5,
        stratify=temp_40_splittable_50_50['section'],
        random_state=seed,
    )

    train_df = pd.concat(
        [train_60, df_non_stratifiable_initial, df_non_stratifiable_from_temp],
        ignore_index=True,
    )
    assert len(train_df) + len(val_20) + len(test_20) == len(df_files)
    return (
        train_df.reset_index(drop=True),
        val_20.reset_index(drop=True),
        test_20.reset_index(drop=True),
    )


def _split_membership(df):
    return set(df['file_path'].apply(_relative_path_from_file_path))


def verify_loaded_splits_match_recreated(train_df, val_df, test_df):
    recreated = dict(zip(
        ['train', 'val', 'test'],
        create_fixed_split(SPLIT_SEED),
    ))
    loaded = {'train': train_df, 'val': val_df, 'test': test_df}
    for split_name in ['train', 'val', 'test']:
        loaded_membership = _split_membership(loaded[split_name])
        recreated_membership = _split_membership(recreated[split_name])
        if loaded_membership != recreated_membership:
            raise RuntimeError(
                f'Loaded {split_name}_split.csv does not match the reported split reconstruction.'
            )


def repair_path(path_value):
    raw = str(path_value)
    path = Path(raw)
    if path.exists():
        return str(path)
    parts = path.parts
    for marker in ['Finished', 'samples']:
        if marker in parts:
            candidate = FINISHED_DIR.joinpath(*parts[parts.index(marker) + 1:])
            if candidate.exists():
                return str(candidate)
    return raw


def load_or_create_splits():
    paths = {name: SPLIT_DIR / f'{name}_split.csv' for name in ['train', 'val', 'test']}
    if all(path.exists() for path in paths.values()):
        print('Loading local split CSVs from', SPLIT_DIR)
        train_df = pd.read_csv(paths['train'])
        val_df = pd.read_csv(paths['val'])
        test_df = pd.read_csv(paths['test'])
        verify_loaded_splits_match_recreated(train_df, val_df, test_df)
    else:
        print('Reconstructing the reported split with split seed', SPLIT_SEED)
        train_df, val_df, test_df = create_fixed_split(SPLIT_SEED)
        train_df.to_csv(paths['train'], index=False)
        val_df.to_csv(paths['val'], index=False)
        test_df.to_csv(paths['test'], index=False)
    for df in [train_df, val_df, test_df]:
        df['file_path'] = df['file_path'].apply(repair_path)
    return train_df, val_df, test_df


train_df, val_df, test_df = load_or_create_splits()
split_sizes = (len(train_df), len(val_df), len(test_df))
split_category_counts = tuple(
    df['section'].nunique() for df in [train_df, val_df, test_df]
)
assert split_sizes == (308, 101, 102), split_sizes
assert split_category_counts == (25, 23, 23), split_category_counts
print('Split sizes:', split_sizes)
print('Category counts:', split_category_counts)


## Load Text And Labels

This cell reads the JSON files referenced by the split dataframes and extracts the text fields used by the models.

Expected JSON fields:

- `input1` through `input6`: clinical-note text components concatenated into the model input.
- `output`: class label.

Labels are encoded once from the training labels, then reused for validation and test labels. If the validation or test split contains a label not seen in training, `LabelEncoder` will fail, which is the correct behavior for this experiment.


In [ ]:
def load_texts_and_labels(df):
    texts=[]; labels=[]; missing=[]
    for _, row in df.iterrows():
        fp=Path(row['file_path'])
        if not fp.exists():
            missing.append(str(fp)); continue
        with open(fp, 'r', encoding='utf-8') as f:
            data=json.load(f)
        parts=[]
        for key in ['input1','input2','input3','input4','input5','input6']:
            value=data.get(key)
            if isinstance(value, str) and value.strip() and value.strip().lower() != 'n/a':
                parts.append(value.strip())
        texts.append(' '.join(parts)); labels.append(row['section'])
    if missing:
        raise FileNotFoundError(f'{len(missing)} missing files. First: {missing[0]}')
    return texts, labels

X_train_text,y_train=load_texts_and_labels(train_df)
X_val_text,y_val=load_texts_and_labels(val_df)
X_test_text,y_test=load_texts_and_labels(test_df)
label_encoder=LabelEncoder()
y_train_encoded=label_encoder.fit_transform(y_train)
y_val_encoded=label_encoder.transform(y_val)
y_test_encoded=label_encoder.transform(y_test)
disease_categories=', '.join(label_encoder.classes_)

assert len(X_train_text)==len(y_train_encoded)
assert len(X_val_text)==len(y_val_encoded)
assert len(X_test_text)==len(y_test_encoded)
assert all(Path(p).exists() for p in train_df['file_path'])
assert all(Path(p).exists() for p in val_df['file_path'])
assert all(Path(p).exists() for p in test_df['file_path'])
print('Loaded:', len(X_train_text), len(X_val_text), len(X_test_text), 'classes:', len(label_encoder.classes_))
print(label_encoder.classes_)

## Feature Helpers And Original Features

## Original TF-IDF Features

This section fits `original_vectorizer` on the training clinical notes only, then transforms validation and test notes with the same vocabulary.

These features are used by `Baseline 1`, `Baseline 1+2b`, and `Baseline 1+2b+3b.1+3bb`.

Do not reuse this vectorizer for LLM-generated features. The LLM summaries have a different text distribution, so they get their own per-seed vectorizer inside the experiment loop.


In [ ]:
def safe_name(value):
    return ''.join(c if c.isalnum() or c in ('-', '_', '.') else '_' for c in str(value))


def text_hash(texts):
    payload = json.dumps(list(texts), ensure_ascii=False, separators=(',', ':'))
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


def cache_metadata_path(array_path):
    return array_path.with_suffix(array_path.suffix + '.metadata.json')


def validate_array_cache_metadata(array_path, expected_metadata):
    metadata_path = cache_metadata_path(array_path)
    if not metadata_path.exists():
        raise ValueError(
            f'Cache metadata missing for {array_path}. Delete the stale cache or regenerate it with this notebook.'
        )
    found = json.loads(metadata_path.read_text(encoding='utf-8'))
    mismatches = {
        key: {'expected': expected_value, 'found': found.get(key)}
        for key, expected_value in expected_metadata.items()
        if found.get(key) != expected_value
    }
    if mismatches:
        raise ValueError(
            f'Cache metadata mismatch for {array_path}: {json.dumps(mismatches, ensure_ascii=False, indent=2)}'
        )
    arr = np.load(array_path, allow_pickle=False)
    recorded_shape = found.get('shape')
    if recorded_shape != list(arr.shape):
        raise ValueError(
            f'Cache shape metadata mismatch for {array_path}: metadata={recorded_shape}, array={list(arr.shape)}'
        )
    return arr


def load_or_compute_array(name, compute_fn, metadata):
    path = CACHE_ROOT / f'{safe_name(name)}.npy'
    expected_metadata = dict(metadata)
    expected_metadata['cache_name'] = safe_name(name)
    if path.exists():
        print('Loading cache:', path)
        return validate_array_cache_metadata(path, expected_metadata)

    arr = compute_fn()
    np.save(path, arr)
    metadata_to_write = dict(expected_metadata)
    metadata_to_write['shape'] = list(arr.shape)
    cache_metadata_path(path).write_text(json.dumps(metadata_to_write, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Saved cache:', path)
    return arr


print('Building original clinical-note TF-IDF features...')
original_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = original_vectorizer.fit_transform(X_train_text)
X_val_tfidf = original_vectorizer.transform(X_val_text)
X_test_tfidf = original_vectorizer.transform(X_test_text)

X_train_tfidf_dense = X_train_tfidf.toarray()
X_val_tfidf_dense = X_val_tfidf.toarray()
X_test_tfidf_dense = X_test_tfidf.toarray()
print(X_train_tfidf.shape, X_val_tfidf.shape, X_test_tfidf.shape)


## Original ClinicalBERT Embeddings

This section builds or loads ClinicalBERT embeddings from the original clinical notes for `Baseline 2a`.

The notebook uses the `[CLS]` token embedding as the document representation, matching the old experiment notebook.


In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

CLINICALBERT_MODEL_NAME = 'emilyalsentzer/Bio_ClinicalBERT'


def get_bert_embeddings(texts, tokenizer, model, batch_size=32):
    model.eval(); chunks=[]
    for i in tqdm(range(0, len(texts), batch_size), desc='ClinicalBERT embeddings'):
        batch=texts[i:i+batch_size]
        encoded=tokenizer(batch, padding=True, truncation=True, return_tensors='pt', max_length=512)
        with torch.no_grad():
            outputs=model(input_ids=encoded['input_ids'].to(device), attention_mask=encoded['attention_mask'].to(device))
            chunks.append(outputs.last_hidden_state[:,0,:].cpu().numpy())
    return np.vstack(chunks)


print('Loading ClinicalBERT...')
bert_tokenizer=AutoTokenizer.from_pretrained(CLINICALBERT_MODEL_NAME)
bert_model=AutoModel.from_pretrained(CLINICALBERT_MODEL_NAME).to(device)
X_train_bert=load_or_compute_array(
    'clinicalbert_train',
    lambda: get_bert_embeddings(X_train_text, bert_tokenizer, bert_model),
    {'feature_type': 'clinicalbert_original', 'model_name': CLINICALBERT_MODEL_NAME, 'split_name': 'train', 'source_texts_hash': text_hash(X_train_text)},
)
X_val_bert=load_or_compute_array(
    'clinicalbert_val',
    lambda: get_bert_embeddings(X_val_text, bert_tokenizer, bert_model),
    {'feature_type': 'clinicalbert_original', 'model_name': CLINICALBERT_MODEL_NAME, 'split_name': 'val', 'source_texts_hash': text_hash(X_val_text)},
)
X_test_bert=load_or_compute_array(
    'clinicalbert_test',
    lambda: get_bert_embeddings(X_test_text, bert_tokenizer, bert_model),
    {'feature_type': 'clinicalbert_original', 'model_name': CLINICALBERT_MODEL_NAME, 'split_name': 'test', 'source_texts_hash': text_hash(X_test_text)},
)
del bert_model
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('ClinicalBERT:', X_train_bert.shape, X_val_bert.shape, X_test_bert.shape)


## Original MPNet Embeddings

This section builds or loads MPNet sentence embeddings from the original clinical notes. These features are used in `Baseline 2b`, `Baseline 1+2b`, `Baseline 2b+3bb`, and `Baseline 1+2b+3b.1+3bb`.

The original-note embedding cache does not depend on rerun seed because the source texts are fixed and the embedding model is deterministic for this use.


In [ ]:
MPNET_MODEL_NAME = 'all-mpnet-base-v2'
print('Loading MPNet...')
model_mpnet=SentenceTransformer(MPNET_MODEL_NAME, device=str(device))


def get_mpnet_embeddings(texts, cache_id, split_name, feature_type='mpnet_original', extra_metadata=None, batch_size=32):
    # Note: the MPNet model only supports up to 384 tokens.
    # Longer texts will be truncated, which may cause information loss.
    # For longer notes and llm-based features, we should use a model that supports longer contexts or implementing a strategy to handle long texts (e.g., chunking and pooling).
    metadata = {
        'feature_type': feature_type,
        'model_name': MPNET_MODEL_NAME,
        'split_name': split_name,
        'source_texts_hash': text_hash(texts),
    }
    if extra_metadata:
        metadata.update(extra_metadata)
    return load_or_compute_array(
        f'mpnet_{cache_id}',
        lambda: model_mpnet.encode(texts, batch_size=batch_size, show_progress_bar=True),
        metadata,
    )


X_train_mpnet=get_mpnet_embeddings(X_train_text, 'original_train', 'train')
X_val_mpnet=get_mpnet_embeddings(X_val_text, 'original_val', 'val')
X_test_mpnet=get_mpnet_embeddings(X_test_text, 'original_test', 'test')
print('MPNet:', X_train_mpnet.shape, X_val_mpnet.shape, X_test_mpnet.shape)


## Prompt_V3 LLM Features

## Prompt_V3 LLM Feature Cache

The LLM cache stores one file per split and rerun seed. Cache filenames include the model, prompt name, seed, and split, for example:

`llm_features_gpt-4o-mini_Prompt_V3_seed42_train.json`

Each cache file also includes metadata:

- OpenAI model name
- prompt name
- prompt hash
- hash of the source texts
- seed
- split name
- temperature

If any metadata does not match the current notebook settings, the notebook raises an error instead of silently using stale LLM features. This is intentional: stale cache reuse is a serious research risk.

This section intentionally uses the canonical Prompt_V3 copied from `Bản sao của Bản sao của Experiment - new codename.ipynb`. The prompt depends on `disease_categories = ', '.join(label_encoder.classes_)`, which is created after labels are encoded. Because the prompt text is part of the cache metadata hash, caches generated with a different Prompt_V3 text will fail validation instead of being reused.

Raw LLM caches and derived LLM embedding caches are both prompt-sensitive. The raw LLM cache validates `prompt_hash`, and the LLM MPNet `.npy` cache stores sidecar metadata with the same `prompt_hash` and `run_seed`.


In [ ]:
PROMPT_NAME = 'Prompt_V3'
PROMPT_TEMPLATE_V3 = f'''Analyze the following clinical text to provide a diagnostic summary and identify key diagnostic phrases.

**Instructions:**
1.  **Diagnostic Summary:** Provide a concise diagnostic summary, including a brief reasoning based on symptoms, lab results, and patient history mentioned in the text.
2.  **Key Diagnostic Phrases:** List all relevant key diagnostic phrases extracted from the text. Ensure these phrases are highly relevant to one or more of the following disease categories: {disease_categories}.
3.  **Format:** Respond only with the diagnostic summary (including reasoning) and the list of key diagnostic phrases, and no other introductory or concluding text.

**Clinical Text:** {{text}}'''


def prompt_hash(prompt_template):
    return hashlib.sha256(prompt_template.encode('utf-8')).hexdigest()


def llm_cache_path(split_name, seed):
    return CACHE_ROOT / f'llm_features_{OPENAI_MODEL}_{PROMPT_NAME}_seed{seed}_{split_name}.json'


def validate_llm_cache(payload, texts, split_name, seed):
    expected = {
        'model': OPENAI_MODEL,
        'prompt_name': PROMPT_NAME,
        'prompt_hash': prompt_hash(PROMPT_TEMPLATE_V3),
        'texts_hash': text_hash(texts),
        'seed': seed,
        'split_name': split_name,
        'temperature': OPENAI_TEMPERATURE,
        'max_tokens': OPENAI_MAX_TOKENS,
    }
    mismatches = {
        key: {'expected': expected_value, 'found': payload.get(key)}
        for key, expected_value in expected.items()
        if payload.get(key) != expected_value
    }
    if mismatches:
        raise ValueError(
            f'LLM cache metadata mismatch for {split_name} seed {seed}: {json.dumps(mismatches, ensure_ascii=False, indent=2)}'
        )
    features = payload.get('features')
    if not isinstance(features, list) or len(features) != len(texts):
        raise ValueError(
            f'LLM cache feature count mismatch for {split_name} seed {seed}: expected {len(texts)}, found {0 if features is None else len(features)}'
        )


def _get_single_llm_feature_openai(text, prompt_template, model_name, seed, max_attempts=3, delay_between_attempts=5):
    require_external_llm_data_compliance()
    generated_feature = 'Error: OpenAI API call failed'
    prompt_tokens_count = 0
    completion_tokens_count = 0
    for attempt in range(max_attempts):
        try:
            messages = [{'role': 'user', 'content': prompt_template.format(text=text)}]
            response = openai_client.chat.completions.create(
                model=model_name,
                messages=messages,
                max_tokens=OPENAI_MAX_TOKENS,
                temperature=OPENAI_TEMPERATURE,
                seed=seed,
            )
            if response.choices and response.choices[0].message.content:
                generated_feature = response.choices[0].message.content.strip()
                if response.usage:
                    prompt_tokens_count = response.usage.prompt_tokens
                    completion_tokens_count = response.usage.completion_tokens
                return generated_feature, prompt_tokens_count, completion_tokens_count
            print(f'Warning: Empty or malformed response for text, attempt {attempt + 1}. Retrying...')
            time.sleep(delay_between_attempts)
        except openai.APITimeoutError:
            print(f'Timeout error for text, attempt {attempt + 1}. Retrying in {delay_between_attempts}s...')
            time.sleep(delay_between_attempts)
        except openai.APIConnectionError as exc:
            print(f'Connection error for text, attempt {attempt + 1}: {exc}. Retrying in {delay_between_attempts}s...')
            time.sleep(delay_between_attempts)
        except openai.RateLimitError as exc:
            if 'insufficient_quota' in str(exc):
                print(f'Quota error: {exc}. Please check your OpenAI plan and rerun. Skipping retries for this sample.')
                return 'Error: Insufficient Quota', 0, 0
            print(f'Rate limit hit for text, attempt {attempt + 1}. Waiting longer ({delay_between_attempts * 2}s) before retrying...')
            time.sleep(delay_between_attempts * 2)
        except Exception as exc:
            print(f'Error generating LLM feature for text, attempt {attempt + 1}: {exc}. Retrying in {delay_between_attempts}s...')
            time.sleep(delay_between_attempts)
    return generated_feature, prompt_tokens_count, completion_tokens_count


def get_llm_features_openai(texts, prompt_template, model_name, split_name, seed, cache_filename=None, max_workers=3, max_attempts=3, delay_between_attempts=5, force_refresh=False):
    if cache_filename is None:
        cache_filename = llm_cache_path(split_name, seed)
    cache_filename = Path(cache_filename)

    if cache_filename.exists() and not force_refresh:
        payload = json.loads(cache_filename.read_text(encoding='utf-8'))
        validate_llm_cache(payload, texts, split_name, seed)
        print(f'Loaded {split_name} LLM features for seed {seed}:', cache_filename)
        return payload['features']

    llm_features = [None] * len(texts)
    items = [None] * len(texts)
    total_prompt_tokens = 0
    total_completion_tokens = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_index = {
            executor.submit(
                _get_single_llm_feature_openai,
                text,
                prompt_template,
                model_name,
                seed,
                max_attempts,
                delay_between_attempts,
            ): i
            for i, text in enumerate(texts)
        }
        for future in tqdm(concurrent.futures.as_completed(future_to_index), total=len(texts), desc=f'{split_name} LLM seed {seed}'):
            index = future_to_index[future]
            try:
                feature, prompt_tokens, completion_tokens = future.result()
                llm_features[index] = feature
                total_prompt_tokens += prompt_tokens
                total_completion_tokens += completion_tokens
                items[index] = {
                    'index': index,
                    'feature': feature,
                    'prompt_tokens': prompt_tokens,
                    'completion_tokens': completion_tokens,
                }
            except Exception as exc:
                feature = 'Error: Parallel processing exception'
                llm_features[index] = feature
                items[index] = {'index': index, 'feature': feature, 'error': repr(exc)}

    payload = {
        'model': OPENAI_MODEL,
        'prompt_name': PROMPT_NAME,
        'prompt_hash': prompt_hash(PROMPT_TEMPLATE_V3),
        'texts_hash': text_hash(texts),
        'seed': seed,
        'split_name': split_name,
        'temperature': OPENAI_TEMPERATURE,
        'max_tokens': OPENAI_MAX_TOKENS,
        'features': llm_features,
        'items': items,
        'prompt_tokens': total_prompt_tokens,
        'completion_tokens': total_completion_tokens,
    }
    cache_filename.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Saved {split_name} LLM features for seed {seed}:', cache_filename)
    return llm_features


def get_llm_features(texts, split_name, seed, force_refresh=False):
    return get_llm_features_openai(
        texts,
        PROMPT_TEMPLATE_V3,
        OPENAI_MODEL,
        split_name,
        seed,
        cache_filename=llm_cache_path(split_name, seed),
        max_workers=3,
        max_attempts=3,
        delay_between_attempts=5,
        force_refresh=force_refresh,
    )


def assert_no_llm_errors(features, split_name, seed):
    errors = [i for i, value in enumerate(features) if str(value).startswith('Error:')]
    if errors:
        error_path = RESULTS_ROOT / f'llm_errors_{PROMPT_NAME}_seed{seed}_{split_name}.json'
        error_path.write_text(json.dumps({'split': split_name, 'seed': seed, 'error_indices': errors}, indent=2), encoding='utf-8')
        raise RuntimeError(f'Found {len(errors)} LLM errors for {split_name} seed {seed}. Saved details to {error_path}')


## Evaluation Helpers

## Evaluation Details

All methods use the same XGBoost configuration, with only `random_state` changing by rerun seed.

Reported metrics:

- `accuracy`
- `f1_micro`
- `f1_macro`

The F1 metrics use standard sklearn behavior with `zero_division=0`. Classification reports are generated with all labels known from the training set so reports remain comparable across validation and test splits.


In [ ]:
def make_xgb(seed):
    return xgb.XGBClassifier(
        objective='multi:softmax',
        num_class=len(label_encoder.classes_),
        eval_metric='mlogloss',
        use_label_encoder=False,
        enable_categorical=True,
        n_estimators=200,
        learning_rate=0.05,
        random_state=seed
    )


def evaluate_xgb_method(method_name, feature_inputs, train_X, val_X, test_X, run_seed, prompt_template='Prompt_V3'):
    model = make_xgb(run_seed)
    model.fit(train_X, y_train_encoded)

    row = {
        'method': method_name,
        'feature_inputs': feature_inputs,
        'prompt_template': prompt_template,
        'split_seed': SPLIT_SEED,
        'run_seed': run_seed,
        'xgb_random_state': run_seed,
        'llm_seed': run_seed,
        'n_train': len(y_train_encoded),
        'n_val': len(y_val_encoded),
        'n_test': len(y_test_encoded),
    }

    reports = {}
    all_labels = np.arange(len(label_encoder.classes_))
    all_target_names = label_encoder.classes_
    for split_name, X, y in [('val', val_X, y_val_encoded), ('test', test_X, y_test_encoded)]:
        pred = model.predict(X)
        row[f'{split_name}_accuracy'] = accuracy_score(y, pred)
        row[f'{split_name}_f1_micro'] = f1_score(y, pred, average='micro', zero_division=0)
        row[f'{split_name}_f1_macro'] = f1_score(y, pred, average='macro', zero_division=0)
        reports[split_name] = classification_report(
            y,
            pred,
            labels=all_labels,
            target_names=all_target_names,
            output_dict=True,
            zero_division=0,
        )
    return row, reports


def require_same_rows(name, X, expected_rows):
    if X.shape[0] != expected_rows:
        raise ValueError(f'{name} has {X.shape[0]} rows, expected {expected_rows}')


def summarize_results(results_df):
    metrics = [
        'val_accuracy', 'test_accuracy',
        'val_f1_micro', 'test_f1_micro',
        'val_f1_macro', 'test_f1_macro',
    ]
    rows = []
    for (method, feature_inputs, prompt_template), group in results_df.groupby(
        ['method', 'feature_inputs', 'prompt_template'], sort=False
    ):
        row = {
            'method': method,
            'feature_inputs': feature_inputs,
            'prompt_template': prompt_template,
            'n_runs': len(group),
        }
        for metric in metrics:
            row[f'{metric}_mean'] = group[metric].mean()
            row[f'{metric}_std'] = group[metric].std(ddof=1)
        rows.append(row)
    return pd.DataFrame(rows).sort_values('test_accuracy_mean', ascending=False).reset_index(drop=True)


## Run Experiments

## Run All Methods Across Seeds

This is the main experiment loop. For each rerun seed, it performs these steps:

1. Load or generate Prompt_V3 LLM features for train, validation, and test.
2. Stop immediately if any LLM feature row starts with `Error:`.
3. Fit a fresh TF-IDF vectorizer on the training LLM features for that seed.
4. Build MPNet embeddings for the LLM features, using seed-specific cache names.
5. Train and evaluate all nine XGBoost methods.

The feature concatenation order follows the method names exactly. For example, `Baseline 1+2b+3b.1+3bb` is built as TF-IDF(original), MPNet(original), TF-IDF(LLM), then MPNet(LLM).

This cell is the expensive one because it may call the OpenAI API and compute embeddings. Re-running it with existing valid caches should be much faster.


In [ ]:
all_result_rows = []
all_reports = {}

for run_seed in RUN_SEEDS:
    print(f'\n=== RUN SEED {run_seed} ===')

    train_llm = get_llm_features(X_train_text, 'train', run_seed)
    val_llm = get_llm_features(X_val_text, 'val', run_seed)
    test_llm = get_llm_features(X_test_text, 'test', run_seed)
    assert_no_llm_errors(train_llm, 'train', run_seed)
    assert_no_llm_errors(val_llm, 'val', run_seed)
    assert_no_llm_errors(test_llm, 'test', run_seed)

    # Correctness-first design: TF-IDF(LLM) has its own vocabulary fitted on LLM-derived training features.
    llm_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
    X_train_llm_tfidf = llm_vectorizer.fit_transform(train_llm)
    X_val_llm_tfidf = llm_vectorizer.transform(val_llm)
    X_test_llm_tfidf = llm_vectorizer.transform(test_llm)
    X_train_llm_tfidf_dense = X_train_llm_tfidf.toarray()
    X_val_llm_tfidf_dense = X_val_llm_tfidf.toarray()
    X_test_llm_tfidf_dense = X_test_llm_tfidf.toarray()

    llm_mpnet_metadata = {'prompt_name': PROMPT_NAME, 'prompt_hash': prompt_hash(PROMPT_TEMPLATE_V3), 'run_seed': run_seed}
    X_train_llm_mpnet = get_mpnet_embeddings(train_llm, f'llm_mpnet_{PROMPT_NAME}_seed{run_seed}_train', 'train', 'mpnet_llm_features', llm_mpnet_metadata)
    X_val_llm_mpnet = get_mpnet_embeddings(val_llm, f'llm_mpnet_{PROMPT_NAME}_seed{run_seed}_val', 'val', 'mpnet_llm_features', llm_mpnet_metadata)
    X_test_llm_mpnet = get_mpnet_embeddings(test_llm, f'llm_mpnet_{PROMPT_NAME}_seed{run_seed}_test', 'test', 'mpnet_llm_features', llm_mpnet_metadata)

    method_specs = [
        (
            'Baseline 1',
            'TF-IDF(original clinical notes)',
            X_train_tfidf,
            X_val_tfidf,
            X_test_tfidf,
        ),
        (
            'Baseline 2a',
            'ClinicalBERT(original clinical notes)',
            X_train_bert,
            X_val_bert,
            X_test_bert,
        ),
        (
            'Baseline 2b',
            'MPNet(original clinical notes)',
            X_train_mpnet,
            X_val_mpnet,
            X_test_mpnet,
        ),
        (
            'Baseline 3b.1',
            'TF-IDF(LLM features)',
            X_train_llm_tfidf,
            X_val_llm_tfidf,
            X_test_llm_tfidf,
        ),
        (
            'Baseline 1+2b',
            'TF-IDF(original clinical notes) + MPNet(original clinical notes)',
            np.hstack((X_train_tfidf_dense, X_train_mpnet)),
            np.hstack((X_val_tfidf_dense, X_val_mpnet)),
            np.hstack((X_test_tfidf_dense, X_test_mpnet)),
        ),
        (
            'Baseline 3bb',
            'MPNet(LLM features)',
            X_train_llm_mpnet,
            X_val_llm_mpnet,
            X_test_llm_mpnet,
        ),
        (
            'Baseline 3b.1+3bb',
            'TF-IDF(LLM features) + MPNet(LLM features)',
            np.hstack((X_train_llm_tfidf_dense, X_train_llm_mpnet)),
            np.hstack((X_val_llm_tfidf_dense, X_val_llm_mpnet)),
            np.hstack((X_test_llm_tfidf_dense, X_test_llm_mpnet)),
        ),
        (
            'Baseline 2b+3bb',
            'MPNet(original clinical notes) + MPNet(LLM features)',
            np.hstack((X_train_mpnet, X_train_llm_mpnet)),
            np.hstack((X_val_mpnet, X_val_llm_mpnet)),
            np.hstack((X_test_mpnet, X_test_llm_mpnet)),
        ),
        (
            'Baseline 1+2b+3b.1+3bb',
            'TF-IDF(original clinical notes) + MPNet(original clinical notes) + TF-IDF(LLM features) + MPNet(LLM features)',
            np.hstack((X_train_tfidf_dense, X_train_mpnet, X_train_llm_tfidf_dense, X_train_llm_mpnet)),
            np.hstack((X_val_tfidf_dense, X_val_mpnet, X_val_llm_tfidf_dense, X_val_llm_mpnet)),
            np.hstack((X_test_tfidf_dense, X_test_mpnet, X_test_llm_tfidf_dense, X_test_llm_mpnet)),
        ),
    ]

    seed_rows = []
    for method_name, feature_inputs, train_X, val_X, test_X in method_specs:
        require_same_rows(f'{method_name} train_X', train_X, len(y_train))
        require_same_rows(f'{method_name} val_X', val_X, len(y_val))
        require_same_rows(f'{method_name} test_X', test_X, len(y_test))
        row, reports = evaluate_xgb_method(method_name, feature_inputs, train_X, val_X, test_X, run_seed)
        all_result_rows.append(row)
        seed_rows.append(row)
        all_reports[f'{method_name}__seed{run_seed}'] = reports
        print(f"{method_name}: val_acc={row['val_accuracy']:.4f}, test_acc={row['test_accuracy']:.4f}")

    assert len(seed_rows) == 9, f'Expected 9 method rows for seed {run_seed}, got {len(seed_rows)}'

results_df = pd.DataFrame(all_result_rows)
assert len(results_df) == 9 * len(RUN_SEEDS), f'Expected 27 detailed rows, got {len(results_df)}'
results_df


## Save Results

## Output Files

This section writes the experiment outputs to `RESULTS_ROOT`, which is usually `/content/drive/MyDrive/NCKH/results/` in Colab.

Files written:

- `rerun_xgboost_methods_3seeds_detailed.csv`: one row per method per run seed.
- `rerun_xgboost_methods_3seeds_summary.csv`: mean and standard deviation by method.
- `rerun_xgboost_methods_3seeds_results.xlsx`: Excel version with `detailed` and `summary` sheets, if `openpyxl` is available.

Use the detailed CSV for auditability and the summary CSV/Excel sheet for reporting mean and standard deviation in the paper.


In [ ]:
detailed_path=RESULTS_ROOT / 'rerun_xgboost_methods_3seeds_detailed.csv'
summary_path=RESULTS_ROOT / 'rerun_xgboost_methods_3seeds_summary.csv'
excel_path=RESULTS_ROOT / 'rerun_xgboost_methods_3seeds_results.xlsx'

results_df.to_csv(detailed_path, index=False)
summary_df=summarize_results(results_df)
summary_df.to_csv(summary_path, index=False)
try:
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        results_df.to_excel(writer, sheet_name='detailed', index=False)
        summary_df.to_excel(writer, sheet_name='summary', index=False)
    print('Saved Excel:', excel_path)
except Exception as exc:
    print('Excel export skipped:', exc)

print('Saved detailed CSV:', detailed_path)
print('Saved summary CSV:', summary_path)
print('Rows:', len(results_df), 'Summary rows:', len(summary_df))
display(summary_df)

## Final Sanity Checks

This final cell verifies that the expected experiment shape was produced:

- `27` detailed rows: `9` methods times `3` rerun seeds.
- `9` summary rows: one per method.
- The recorded run seeds match `[42, 123, 456]`.

If any assertion fails, do not use the result files until the cause is understood.


In [ ]:
print('Detailed rows:', len(results_df))
print('Summary rows:', len(summary_df))
print('Methods:', sorted(results_df['method'].unique()))
print('Seeds:', sorted(results_df['run_seed'].unique()))
assert len(results_df) == 27
assert len(summary_df) == 9
assert sorted(results_df['run_seed'].unique()) == RUN_SEEDS
metric_cols = [
    'val_accuracy', 'test_accuracy',
    'val_f1_micro', 'test_f1_micro',
    'val_f1_macro', 'test_f1_macro',
]
assert np.isfinite(results_df[metric_cols].to_numpy()).all()
assert ((results_df[metric_cols] >= 0) & (results_df[metric_cols] <= 1)).all().all()
assert (summary_df['n_runs'] == len(RUN_SEEDS)).all()
print('Final checks passed.')
